In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import gaussian_kde

from scipy.ndimage import gaussian_filter

In [ ]:
# read all_samples as in 6a--MCMC_Analysis.ipynb

In [ ]:
samples = all_samples

# --------------------------
# SETUP FIGURE
# --------------------------
fig, axes = plt.subplots(ndim, ndim, figsize=(4*ndim, 4*ndim))
plt.subplots_adjust(wspace=0.15, hspace=0.15, bottom=0.1, left=0.1, top=0.85, right=0.85)

for i in range(ndim):
    for j in range(ndim):
        ax = axes[i, j]
        ax.set_xlim(ranges[j])
        ax.set_ylim(ranges[i])
        # remove tick labels inside
        if i != ndim-1:
            ax.xaxis.set_ticklabels([])
        if j != 0:
            ax.yaxis.set_ticklabels([])

# --------------------------
# 2D HISTOGRAMS / KDE
# --------------------------
for i in range(1, ndim):
    for j in range(i):
        x = samples[:, j]
        y = samples[:, i]
        ax = axes[i, j]
        
        H, xedges, yedges = np.histogram2d(x, y, bins=40, range=[ranges[j], ranges[i]])
        H_smooth = gaussian_filter(H, sigma=0.75)

        Xc = 0.5*(xedges[:-1] + xedges[1:])
        Yc = 0.5*(yedges[:-1] + yedges[1:])
        im = ax.imshow(H_smooth.T, 
                       origin='lower', 
                       extent=(ranges[j][0], ranges[j][1], ranges[i][0], ranges[i][1]),
                       aspect='auto', 
                       cmap='Greys_r', 
                       norm=mcolors.LogNorm(vmin=3e-1*H.max(), vmax=H.max())
                       #norm=mcolors.Normalize(vmin=3e-1*H.max(), vmax=H.max())
                       )
        
        ax.contour(Xc, Yc, H_smooth.T, levels=[H.max()*f for f in (0.16,0.3, 0.5, 0.7, 0.8, 0.9, )], colors='k')

        # truth marker
        ax.plot(best_values[j], best_values[i], marker='X', markersize=12, c='k')

# --------------------------
# 1D HISTOGRAMS + PRIOR CURVES
# --------------------------
for i in range(ndim):
    ax = axes[i, i]
    x = samples[:, i]

    n_bins = 25

    # histogram
    counts, bin_edges = np.histogram(x, bins=n_bins, density=True)
    counts /= counts.max() 
    ax.hist(bin_edges[:-1]+1e-6, bins=n_bins, weights=counts, color='grey', linewidth=1.5, histtype='step')

    # prior (from your code)
    xi = np.linspace(ranges[i][0], ranges[i][1], 200)
    dx = xi[1]-xi[0]
    prior = np.exp(-(xi-params_mean[i])**2/(2*params_sigma[i]**2))
    prior /= np.max(prior)
    ax.plot(xi, prior, c='k', ls='--')
    ax.set_ylim(0,1.1)

    # optional truth
    ax.axvline(best_values[i], color='dimgrey', linewidth=2)

# --------------------------
# LABELS
# --------------------------
for i in range(ndim):
    for j in range(ndim):
        if i == ndim-1:
            axes[i, j].set_xlabel(p_names[j], fontsize=28)
        if j == 0:
            axes[i, j].set_ylabel(p_names[i], fontsize=28)

# --------------------------
# SAVE FIGURE
# --------------------------
#fig.savefig('posterior_ma